# 13 · Verifica: il device è fermo davvero?
*Blocco 3 · Certificati. Passo di verifica, con un ripristino in fondo.*

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

In [ ]:
$id = 'device-da-revocare'
"stato in ADR: " + (Get-AdrDeviceState -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg)

stato in ADR: Disabled

## Anche il certificato valido è rifiutato
Il certificato ottenuto dopo la revoca (notebook 11) è valido, ma il device è disabilitato.

In [ ]:
$nuovo = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '11-device-nuova-registrazione'
Show-HubAcceptance -Label 'certificato valido, device disabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $nuovo.Dir

certificato valido, device disabilitato              HTTP 401  (rifiutato)

## Cosa abbiamo visto
- La revoca invalida i **certificati**; la disabilitazione ferma il **device**.
- Con il device disabilitato, un certificato valido viene rifiutato e una nuova registrazione fallisce.

## Ripristino
Riabilita il device, così i notebook 09-13 si possono rilanciare.

In [ ]:
"stato in ADR: " + (Set-AdrDeviceEnablement -Name $id -Namespace $cfg.AdrName -ResourceGroup $cfg.Rg -State Enabled)

stato in ADR: Enabled

## Dopo la riabilitazione
Il device è di nuovo abilitato. Cosa dice l'Hub dei certificati di prima? Quello **valido** e quello **revocato** (notebook 09).

In [ ]:
$prima = Open-ScenarioDevice -Config $cfg -DeviceId $id -Notebook '09-device-revoca'
Show-HubAcceptance -Label 'certificato valido, device riabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $nuovo.Dir
Show-HubAcceptance -Label 'certificato revocato, device riabilitato' -Hub $nuovo.AssignedHub -DeviceId $id -StateDir $prima.Dir

certificato valido, device riabilitato               HTTP 401  (rifiutato)

certificato revocato, device riabilitato             HTTP 401  (rifiutato)

Da qui la regola per un device compromesso: **prima si disabilita** (ferma subito l'accesso e impedisce di ottenere un certificato nuovo), **poi si revoca** (i vecchi certificati non tornano validi alla riabilitazione).

Prossimo notebook: **14**, gruppi, aggiornamenti e job.